# Conley Spatial HAC Standard Errors

**Econometrics Notebook Library · v0.1.0**

## Intuition

Clustering assumes a block structure: arbitrary dependence within clusters, independence across them. Spatial processes often decay continuously with distance instead. Conley-style spatial HAC estimators allow cross-sectional covariance between observations that are geographically close, with a kernel that downweights distant pairs.

Reference: [Conley, Journal of Econometrics (1999)](https://ideas.repec.org/a/eee/econom/v92y1999i1p1-45.html).

## Sandwich derivation

For OLS $\hat\beta=(X'X)^{-1}X'Y$, write $s_i=x_i\hat u_i$. A spatial HAC meat matrix is

$$
\widehat S=\sum_i\sum_j K\!\left(\frac{d_{ij}}{c}\right)s_i s_j',
$$

where $d_{ij}$ is distance and $c$ is a cutoff. The covariance estimator is

$$
\widehat V=(X'X)^{-1}\widehat S(X'X)^{-1}.
$$

With a Bartlett kernel, $K(r)=\max(1-r,0)$.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (8, 4.5)
pd.set_option("display.max_columns", 30)

In [ ]:
import statsmodels.api as sm
from econnotes.core import simulate_spatial_cross_section, conley_covariance

df = simulate_spatial_cross_section(n=280, seed=91)
X = sm.add_constant(df[["x"]]).to_numpy()
fit = sm.OLS(df.y, X).fit(cov_type="HC1")
coords = df[["xcoord","ycoord"]].to_numpy()
V20 = conley_covariance(X, fit.resid, coords, cutoff=20)
V35 = conley_covariance(X, fit.resid, coords, cutoff=35)
comparison = pd.DataFrame({
    "SE type":["HC1", "Conley c=20", "Conley c=35"],
    "SE(beta_x)":[fit.bse[1], np.sqrt(V20[1,1]), np.sqrt(V35[1,1])]
})
comparison

In [ ]:
fig, ax = plt.subplots()
sc = ax.scatter(df.xcoord, df.ycoord, c=fit.resid, s=28)
fig.colorbar(sc, ax=ax, label="OLS residual")
ax.set(xlabel="X coordinate", ylabel="Y coordinate", title="Residual spatial structure is not a cluster label");

## Cutoff sensitivity is part of the analysis

There is no universally correct kilometer cutoff. The cutoff encodes a substantive claim about how far residual dependence persists. A credible paper reports sensitivity across economically plausible distances and explains the coordinate/distance metric.

## Researcher failure checklist

- Do not feed latitude/longitude degrees into a Euclidean-distance routine and call the cutoff “kilometers.”
- Justify the kernel and cutoff substantively.
- Distinguish spatial correlation in errors from spatial spillovers in treatment; the latter is an identification problem, not merely an SE problem.
- For panel data, decide how spatial and temporal dependence are combined.
- Report sensitivity, especially when significance changes around one arbitrary cutoff.